In [37]:
import pandas as pd
import numpy as np
from statsmodels.tsa.api import VAR
from statsmodels.tsa.ar_model import AutoReg

In [38]:
# change the original data set to different time frequency
def dataframe_alternative_frequency(dataset, time):
    dataset['open_time'] = pd.to_datetime(dataset['open_time'], utc=True)  # <- key fix
    result = (dataset.sort_values('open_time').groupby(dataset['open_time'].dt.to_period(time)).tail(1))
    return result

In [39]:
dfbtc = pd.read_csv('/Users/hedonglin/Desktop/project/crypto/BTCUSDT_1m_20200101_20260119.csv')
dfbtcweek = dataframe_alternative_frequency(dfbtc, 'W')
dfbtcweek['return'] = (np.log(dfbtcweek['close']) - np.log(dfbtcweek['close']).shift(1)) * 52

dfeth = pd.read_csv('/Users/hedonglin/Desktop/project/crypto/ETHUSDT_1m_20200101_20260119.csv')
dfethweek = dataframe_alternative_frequency(dfeth, 'W')
dfethweek['return'] = (np.log(dfethweek['close']) - np.log(dfethweek['close']).shift(1)) * 52

print(dfethweek)

/var/folders/3m/z8zx2xzn17v946lswjlz9t7h0000gn/T/ipykernel_2117/3430624170.py:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  result = (dataset.sort_values('open_time').groupby(dataset['open_time'].dt.to_period(time)).tail(1))


                        open_time     open     high      low    close  \
7199    2020-01-05 23:59:00+00:00   135.37   135.39   135.33   135.33   
17279   2020-01-12 23:59:00+00:00   146.21   146.60   146.21   146.57   
27359   2020-01-19 23:59:00+00:00   167.22   167.29   167.04   167.04   
37439   2020-01-26 23:59:00+00:00   167.83   168.03   167.83   167.90   
47519   2020-02-02 23:59:00+00:00   188.69   188.72   188.40   188.59   
...                           ...      ...      ...      ...      ...   
3162239 2026-01-04 23:59:00+00:00  3141.39  3145.36  3141.12  3143.32   
3172319 2026-01-11 23:59:00+00:00  3122.05  3122.57  3120.45  3122.00   
3182399 2026-01-18 23:59:00+00:00  3283.05  3284.31  3281.27  3282.77   
3192479 2026-01-25 23:59:00+00:00  2816.77  2817.19  2815.30  2815.49   
3196800 2026-01-29 00:00:00+00:00  3009.36  3012.04  3008.87  3010.38   

           volume                        close_time  quote_asset_volume  \
7199      276.106  2020-01-05T23:59:59.999000+00

/var/folders/3m/z8zx2xzn17v946lswjlz9t7h0000gn/T/ipykernel_2117/3430624170.py:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  result = (dataset.sort_values('open_time').groupby(dataset['open_time'].dt.to_period(time)).tail(1))


# The correlation matrix among crypto currencies


In [40]:
a = dfbtcweek['return']
b = dfethweek['return']

b.corr(a)

b.corr(a.shift(1))
b.corr(a.shift(-1))

np.float64(0.08900417034215803)

In [41]:
# btc
a2 = a.reset_index(drop=True)
a3 = a2.dropna()
model = AutoReg(a3, lags=5)
result = model.fit()

print(result.summary())


ValueError: operands could not be broadcast together with shapes (312,) (313,) 

In [42]:
a2.corr(a2.shift(-1))
b2.corr(b2.shift(1))

# eth
b2 = b.reset_index(drop=True)
b3 = b2.dropna()
model = AutoReg(b3, lags=5)
result = model.fit()

print(result.summary())



NameError: name 'b2' is not defined

# AutoRegression

In [ ]:
# a = btc price series (pd.Series)
# b = eth price series (pd.Series)

# 1) Align and clean (same timestamps/rows)
prices = pd.concat([a.rename("btc"), b.rename("eth")],axis=1).dropna()

# (optional) ensure numeric
prices = prices.apply(pd.to_numeric, errors="coerce").dropna()

# 2) Option B: log returns
rets = np.log(prices).diff().dropna()

# (optional) make index simple like you did
rets = rets.reset_index(drop=True)

# 3) Fit VAR(5)
model = VAR(rets)
result = model.fit(5)

print(result.summary())



  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Wed, 11, Mar, 2026
Time:                     17:07:11
--------------------------------------------------------------------
No. of Equations:         2.00000    BIC:                    2.16401
Nobs:                     53.0000    HQIC:                   1.66067
Log likelihood:          -164.081    FPE:                    3.88995
AIC:                      1.34616    Det(Omega_mle):         2.66769
--------------------------------------------------------------------
Results for equation btc
            coefficient       std. error           t-stat            prob
-------------------------------------------------------------------------
const          0.306914         0.229687            1.336           0.181
L1.btc        -0.395548         0.183587           -2.155           0.031
L1.eth         0.009245         0.221678            0.042           0.967
L2.btc         0

/opt/anaconda3/envs/crypto_panel/lib/python3.11/site-packages/pandas/core/internals/blocks.py:347: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)
